# Week 5 — Visualisations
Loads saved evaluation results and generates all five figures:
1. Training curve (loss + F1 over epochs)
2. Per-class F1 bar chart (baseline vs fine-tuned)
3. Confusion matrix (top-N most frequent classes)
4. ROC curves with AUC (one-vs-rest, all 28 classes)
5. Precision-Recall curves (all 28 classes)

Run `eval.ipynb` first to generate the required files in `results/`.

In [ ]:
!pip install matplotlib seaborn scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd gdrive/My\ Drive/Colab\ Notebooks/emotion_project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, average_precision_score

os.makedirs('figures', exist_ok=True)

THRESHOLD  = 0.5
NUM_LABELS = 28
LABEL_NAMES = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

# Load eval results
test_labels     = np.load('results/test_labels.npy').astype(int)
baseline_probs  = np.load('results/test_probs_baseline.npy')
finetuned_probs = np.load('results/test_probs_finetuned.npy')
baseline_pc     = pd.read_csv('results/test_per_class_baseline.csv')
finetuned_pc    = pd.read_csv('results/test_per_class_finetuned.csv')

with open('results/test_metrics_baseline.json')  as f: baseline_overall  = json.load(f)
with open('results/test_metrics_finetuned.json') as f: finetuned_overall = json.load(f)

# Load training history (all saved runs)
run_files = sorted(glob.glob('results/run_*.csv'))
all_runs  = pd.concat([pd.read_csv(f) for f in run_files], ignore_index=True)
print('Loaded runs:', all_runs['run'].unique().tolist())
print('Test labels shape:', test_labels.shape)

## 1. Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for idx, (run_name, group) in enumerate(all_runs.groupby('run')):
    c = colors[idx % len(colors)]
    if 'train_loss' in group.columns:
        axes[0].plot(group['epoch'], group['train_loss'], color=c, linestyle='--',
                     label=f'{run_name} train')
    axes[0].plot(group['epoch'], group['loss'], color=c, linestyle='-',
                 label=f'{run_name} val')
    axes[1].plot(group['epoch'], group['macro_f1'], color=c, linestyle='-',
                 label=f'{run_name} macro F1')
    axes[1].plot(group['epoch'], group['micro_f1'], color=c, linestyle='--',
                 label=f'{run_name} micro F1')

axes[0].set_title('Loss over Epochs', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('F1 over Epochs', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('figures/training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/training_curve.png')

## 2. Per-Class F1 Bar Chart

In [ ]:
# Sort by fine-tuned F1 descending
order        = finetuned_pc.sort_values('f1', ascending=False)['label'].tolist()
baseline_f1  = baseline_pc.set_index('label').loc[order, 'f1'].values
finetuned_f1 = finetuned_pc.set_index('label').loc[order, 'f1'].values

x = np.arange(NUM_LABELS)
w = 0.38

fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(x - w/2, baseline_f1,  width=w, label='Baseline',   color='steelblue',  alpha=0.85)
ax.bar(x + w/2, finetuned_f1, width=w, label='Fine-tuned', color='darkorange', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
ax.set_title('Per-Class F1 Score — Baseline vs Fine-tuned  (sorted by Fine-tuned F1)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Annotate overall macro F1 for reference
ax.axhline(baseline_overall['macro_f1'],  color='steelblue',  linestyle=':', lw=1.2,
           label=f'Baseline macro avg ({baseline_overall["macro_f1"]:.3f})')
ax.axhline(finetuned_overall['macro_f1'], color='darkorange', linestyle=':', lw=1.2,
           label=f'Fine-tuned macro avg ({finetuned_overall["macro_f1"]:.3f})')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/per_class_f1.png')

## 3. Confusion Matrix — Top-N Most Frequent Classes

In [ ]:
TOP_N = 10

# Primary true label = highest-index active label (argmax of multi-hot)
# Primary predicted  = argmax of probabilities
true_primary = np.argmax(test_labels,     axis=1)
pred_primary = np.argmax(finetuned_probs, axis=1)

# Select top-N classes by test support
support    = test_labels.sum(axis=0)
topn_idxs  = np.argsort(support)[::-1][:TOP_N]
topn_names = [LABEL_NAMES[i] for i in topn_idxs]
idx_remap  = {old: new for new, old in enumerate(topn_idxs)}

# Filter samples whose true primary label is in top-N
mask         = np.isin(true_primary, topn_idxs)
true_topn    = np.array([idx_remap[l] for l in true_primary[mask]])
# Predictions outside top-N are mapped to TOP_N ("other")
pred_topn    = np.array([idx_remap.get(p, TOP_N) for p in pred_primary[mask]])

cm = confusion_matrix(true_topn, pred_topn, labels=list(range(TOP_N)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalise

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    f'Confusion Matrix — Top {TOP_N} Classes (counts)'),
    (axes[1], cm_norm, '.2f',  f'Confusion Matrix — Top {TOP_N} Classes (row-normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=topn_names, yticklabels=topn_names, ax=ax,
                annot_kws={'size': 8})
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    ax.set_title(title,              fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/confusion_matrix.png')
print(f'Top-{TOP_N} classes by support:', topn_names)

## 4. ROC Curves — All 28 Classes (one-vs-rest)

In [ ]:
NROWS, NCOLS = 4, 7
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(22, 13))
axes = axes.flatten()

models = [
    (baseline_probs,  'Baseline',   'steelblue'),
    (finetuned_probs, 'Fine-tuned', 'darkorange'),
]

for i, (name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    if test_labels[:, i].sum() == 0:
        ax.text(0.5, 0.5, 'no positives', ha='center', va='center',
                transform=ax.transAxes, fontsize=7)
        ax.set_title(name, fontsize=8)
        continue

    for probs, label, color in models:
        fpr, tpr, _ = roc_curve(test_labels[:, i], probs[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=1.2, label=f'{label} {roc_auc:.2f}')

    ax.plot([0, 1], [0, 1], 'k--', lw=0.7)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(name, fontsize=8)
    ax.legend(fontsize=5.5, loc='lower right', framealpha=0.6)

fig.text(0.5, 0.01, 'False Positive Rate', ha='center', fontsize=12)
fig.text(0.01, 0.5, 'True Positive Rate',  va='center', rotation='vertical', fontsize=12)
fig.suptitle('ROC Curves (AUC) — One-vs-Rest for All 28 Emotion Classes', fontsize=14, y=1.01)

plt.tight_layout()
plt.savefig('figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/roc_curves.png')

## 5. Precision-Recall Curves — All 28 Classes

In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(22, 13))
axes = axes.flatten()

for i, (name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    if test_labels[:, i].sum() == 0:
        ax.text(0.5, 0.5, 'no positives', ha='center', va='center',
                transform=ax.transAxes, fontsize=7)
        ax.set_title(name, fontsize=8)
        continue

    # Chance level = class prevalence
    chance = test_labels[:, i].mean()
    ax.axhline(chance, color='grey', linestyle='--', lw=0.8)

    for probs, label, color in models:
        precision, recall, _ = precision_recall_curve(test_labels[:, i], probs[:, i])
        ap = average_precision_score(test_labels[:, i], probs[:, i])
        ax.plot(recall, precision, color=color, lw=1.2, label=f'{label} AP={ap:.2f}')

    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(name, fontsize=8)
    ax.legend(fontsize=5.5, loc='upper right', framealpha=0.6)

fig.text(0.5, 0.01, 'Recall',    ha='center', fontsize=12)
fig.text(0.01, 0.5, 'Precision', va='center', rotation='vertical', fontsize=12)
fig.suptitle('Precision-Recall Curves (AP) — All 28 Emotion Classes', fontsize=14, y=1.01)

plt.tight_layout()
plt.savefig('figures/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/pr_curves.png')

## Summary — figures saved

In [ ]:
print('figures/ directory:')
for f in sorted(os.listdir('figures')):
    print(f'  {f}')